In [4]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# --- 1. LOAD AND PREPROCESS TRAINING DATA ---
data = np.load("../first_batch_with_labels.npz")
X_raw = data["X"]
y_raw = data["y"]



# Convert to DataFrames
ratings = pd.DataFrame(X_raw, columns=["user", "item", "rating"])
labels = pd.DataFrame(y_raw, columns=["user", "label"])

# Engineer features: Group by user to get 5 specific metrics
user_features = ratings.groupby("user").agg({
    "rating": ["mean", "std", "count", "min", "max"]
})

# Flatten multi-index columns and rename to match model expectations
user_features.columns = [
    "mean_rating",
    "std_rating",
    "num_ratings",
    "min_rating",
    "max_rating"
]

# Fill NaN in std_rating (occurs when num_ratings == 1)
user_features["std_rating"] = user_features["std_rating"].fillna(0)

# Align labels with the aggregated features
labels = labels.set_index("user")
X_train = user_features
y_train = labels.loc[X_train.index]["label"]

# --- 2. TRAIN THE MODEL ---
model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train, y_train)

# --- 3. LOAD AND PREPROCESS PREDICTION DATA ---
data_pred = np.load("subset_training_batch.npz")
X_pred_raw = data_pred["X"]

# Convert raw [user, item, rating] into the same 5 features used in training
df_pred = pd.DataFrame(X_pred_raw, columns=["user", "item", "rating"])
X_pred_processed = df_pred.groupby("user").agg({
    "rating": ["mean", "std", "count", "min", "max"]
})

# Rename columns to match the training set exactly
X_pred_processed.columns = [
    "mean_rating",
    "std_rating",
    "num_ratings",
    "min_rating",
    "max_rating"
]

# Handle NaNs in the prediction set
X_pred_processed["std_rating"] = X_pred_processed["std_rating"].fillna(0)

# --- 4. GENERATE PREDICTIONS ---
# This will now work because X_pred_processed has 5 columns, not 3
scores = model.predict_proba(X_pred_processed)[:, 1]

# Display results
results = pd.DataFrame({
    "user": X_pred_processed.index,
    "probability_score": scores
})
print(results.head())

FileNotFoundError: [Errno 2] No such file or directory: 'subset_training_batch.npz'

In [5]:
import numpy as np
import pandas as pd

# 1. Load Batch 1 Data
# Ensure the filename matches your actual file (e.g., "first_batch.npz")
data_pred = np.load("../second_batch.npz")
X_raw_batch1 = data_pred["X"]

# 2. Convert to DataFrame for easier manipulation
df_batch1 = pd.DataFrame(
    X_raw_batch1, 
    columns=["user", "item", "rating"]
)

# 3. Create Features (Must match the 5 features used in training)
features_batch1 = df_batch1.groupby("user").agg({
    "rating": ["mean", "std", "count", "min", "max"]
})

# Flatten column names to match the model's expected feature names
features_batch1.columns = [
    "mean_rating",
    "std_rating",
    "num_ratings",
    "min_rating",
    "max_rating"
]

# Handle users with only 1 rating (where std would be NaN)
features_batch1 = features_batch1.fillna(0)

# 4. Predict Anomaly Scores
# This gives the probability of the 'positive' class (label 1)
batch1_scores = model.predict_proba(features_batch1)[:, 1]

# 5. Save Submission
# It is often helpful to save the user IDs alongside scores 
# to ensure they stay mapped correctly, but here is the raw array:
np.savez("submission_batch1.npz", predictions=batch1_scores)

print(f"Batch 1 processed. Generated {len(batch1_scores)} predictions.")

Batch 1 processed. Generated 860 predictions.
